# Description

W tym notatniku przeprowadzane są wszelkie eksperymenty, zarówno dla autoenkodera wariacyjnego i nie wariacyjnego, dla wszystkich członów funkcji straty, w wersji z douczaniem i bez (łącznie 12 eksperymentów)

# Imports

In [40]:
%load_ext autoreload
%autoreload 2
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import RichProgressBar
import yaml
import sys
import os
import tqdm
import wandb
import json

sys.path.append('../')  # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.GraphAutoencoder import GraphAutoencoder
from src.models.KlejdaGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from pyprojroot import here

current_dir = os.getcwd()
framspy_path = os.path.abspath(os.path.join(current_dir, '..', 'external', 'framspy'))
if framspy_path not in sys.path:
	sys.path.insert(0, framspy_path)
from FramsticksLib import FramsticksLib
from deap import tools, algorithms
import yaml
from src.deap.deap_setup import prepare_native_toolbox
from src.deap.constraints import is_feasible_fitness_criteria
from src.deap.save_and_load_results import save_genotypes_json
from utils.FramsticksPostProcessor import FramsticksPostProcessor
from utils.FramsticksGraphDataset import FramsticksGraphDataset
import numpy as np
import time

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Pre-processing

In [41]:
project_dir = here()
# Przygotowanie checkpointów nauczonych autoenkoderów
checkpoints_dir = project_dir / 'notebooks' / 'checkpoints' / 'final_checkpoints' / 'klejda'
checkpoint_gae = torch.load(checkpoints_dir / 'gae.ckpt')
checkpoint_vgae = torch.load(checkpoints_dir / 'vgae.ckpt')

In [42]:
# Przygotowanie konfiguracji dla gae
configs_dir = project_dir / 'configs'
config_gae_path = configs_dir / 'klejda_gae_config.yaml'
config_vgae_path = configs_dir / 'klejda_vgae_config.yaml'
with open(config_gae_path) as f:
    config_gae = yaml.safe_load(f)
with open(config_vgae_path) as f:
    config_vgae = yaml.safe_load(f)

In [43]:
# Implementacja nowego operatora mutacji
def autoencoder_mutate(gae, individual, sigma=0.1):
	framsticks_genotype = individual[0]

	x_matrix, a_matrix, _ = FramsticksGraphDataset.parse_f0_to_matrices(framsticks_genotype, evolution_config['max_numparts'])
	x_matrix.unsqueeze_(0)
	a_matrix.unsqueeze_(0)

	gae.eval()

	with torch.no_grad():
		z = gae.encoder_backbone(x_matrix,a_matrix)
		z = gae.fc_z(z)

		noise = torch.randn_like(z) * sigma
		z_mutated = z + noise

		a_prime = gae.decoder_a(z_mutated)
		x_prime = gae.decoder_x(z_mutated, a_prime)

		a_prime.squeeze_(0)
		x_prime.squeeze_(0)

		# 6. Konwersja: Wyjście GAE -> Framsticks
		_, new_framsticks_genotype, _ = postProcessingGaeResult.process(x_prime,a_prime)

		# 7. Zastąpienie starego genotypu nowym
		individual[0] = new_framsticks_genotype

		return individual,

In [44]:
# Przygotowanie środowiska Framsticks oraz DEAP
with open("../configs/final_evolution_config.yaml", 'r') as f:
	evolution_config = yaml.safe_load(f)

frams_lib = FramsticksLib(evolution_config['frams_path'], evolution_config['frams_lib'], evolution_config['sim_file'])

toolbox = prepare_native_toolbox(frams_lib, evolution_config)
# TODO: TO dodać przed uruchomieniem eksperymentu
# toolbox.register("mutate", autoencoder_mutate, gae)
pop = toolbox.population(n=evolution_config['pop_size'])
hof = tools.HallOfFame(evolution_config['hof_size'])

stats = tools.Statistics(lambda ind: ind.fitness.values)
filter_feasible = lambda func, criteria: func(list(filter(is_feasible_fitness_criteria, criteria)))
stats.register("min", lambda fit: filter_feasible(np.min, fit))
stats.register("avg", lambda fit: filter_feasible(np.mean, fit))
stats.register("max", lambda fit: filter_feasible(np.max, fit))

postProcessingGaeResult = FramsticksPostProcessor()

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data

Available objects: ['CheckpointEvent', 'Collision', 'CrCollision', 'Creature', 'CreatureSettings', 'CreatureSignals', 'CreatureSnapshot', 'Dictionary', 'ExpProperties', 'ExpState', 'ExtValue', 'File', 'FunctionReference', 'GenMan', 'GenManStats', 'GenePool', 'GenePools', 'Geno', 'GenoConverters', 'Genotype', 'Interface', 'Joint', 'Loader', 'Math', 'MechJoint', 'MechPart', 'MessageCatcher', 'Model', 'ModelGeometry', 'ModelSymmetry', 'Neuro', 'NeuroClass', 'NeuroClassLibrary', 'NeuroDef', 'NeuroSignals', 'NeuronsSimEnabled', 'ODE', 'Orient', 'Part', 'Population', 'Populations', 'Ref', 'Signal', 'SignalView', 'SimilMeasure', 'SimilMeasureDistribution', 'SimilMeasureGreedy', 'SimilMeasureHungarian', 'Simulator', 'SlaveSimulators', 'StopEvent', 'String', 'UserScripts', 'Vec

# Eksperymenty

## GAE

### Non-cyclic

In [45]:
locality_loss_type = ['parts_num', 'fitness', 'dissimilarity']

config_gae['locality_loss_type'] = locality_loss_type[0]
gae_non_cyclic = GraphAutoencoder(config=config_gae)
gae_non_cyclic.load_state_dict(checkpoint_gae['state_dict'])
gae_non_cyclic.eval()
toolbox.register("mutate", autoencoder_mutate, gae_non_cyclic)

pop, log = algorithms.eaSimple(
	pop, toolbox,
	cxpb=evolution_config['p_xov'], mutpb=evolution_config['p_mut'],
	ngen=evolution_config['generations'], stats=stats, halloffame=hof, verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

gen	nevals	min  	avg  	max  
0  	120   	-0.01	-0.01	-0.01
1  	108   	-0.01	-0.01	-0.01
2  	106   	-0.01	-0.01	-0.01
3  	109   	-0.01	-0.01	-0.01
4  	107   	-0.01	-0.01	-0.01
5  	104   	-0.01	-0.01	-0.01
6  	106   	-0.01	-0.01	-0.01
7  	103   	-0.01	-0.01	-0.01
8  	108   	-0.01	-0.01	-0.01
9  	107   	-0.01	-0.01	-0.01
10 	110   	-0.01	-0.01	-0.01
11 	107   	-0.01	-0.01	-0.01
12 	107   	-0.01	-0.01	-0.01
13 	103   	-0.01	-0.01	-0.01
14 	105   	-0.01	-0.01	-0.01
15 	108   	-0.01	-0.01	-0.01
16 	103   	-0.01	-0.01	-0.01
17 	112   	-0.01	-0.01	-0.01
18 	109   	-0.01	-0.01	-0.01
19 	109   	-0.01	-0.01	-0.01
20 	105   	-0.01	-0.01	-0.01
21 	115   	-0.01	-0.01	-0.01
22 	107   	-0.01	-0.01	-0.01
23 	110   	-0.01	-0.01	-0.01
24 	108   	-0.01	-0.01	-0.01
25 	110   	-0.01	-0.01	-0.01
26 	112   	-0.01	-0.01	-0.01
27 	105   	-0.01	-0.01	-0.01
28 	113   	-0.01	-0.01	-0.01
29 	107   	-0.01	-0.01	-0.01
30 	105   	-0.01	-0.01	-0.01
31 	111   	-0.01	-0.01	-0.01
32 	108   	-0.01	-0.01	-0.01
33 	110   	-0.

### Cyclic

## VGAE

 ### Part number locality loss

### Fitness locality loss

### Dissimilarity locality loss

### Part number locality loss cyclic

### Fitness locality loss cyclic

### Dissimilarity locality loss cyclic

# Results